# nnU-Net 3D full resolution + soft-clDice (v1)

## Experimento 2 — ImageCAS

**Objetivo:** modificar la función de pérdida de nnU-Net para incorporar información topológica mediante soft-clDice.

### Baseline
- Dataset: Dataset501_ImageCAS
- Configuración: 3d_fullres
- Fold: 0
- Épocas: 100
- Pérdida: Dice + Cross-Entropy

### Modelo propuesto
- Misma arquitectura, dataset, fold y número de épocas
- Pérdida: Dice + Cross-Entropy + soft-clDice

\[
L_{total} = L_{Dice+CE} + \lambda L_{soft-clDice}
\]

Valor inicial del peso topológico:

\[
\lambda = 0.2
\]


In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA disponible: True
GPU: NVIDIA A100-SXM4-40GB


In [2]:
!python -m pip install -q nnunetv2

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.1/291.1 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.9/28.9 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 118.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 107.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 116.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 11.9 MB/s eta 0:

In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os
from pathlib import Path

BASE = Path("/content/drive/MyDrive/TFM_nnUNet")

NNUNET_RAW = BASE / "nnUNet_raw"
NNUNET_PREPROCESSED = BASE / "nnUNet_preprocessed"
NNUNET_RESULTS = BASE / "nnUNet_results"

os.environ["nnUNet_raw"] = str(NNUNET_RAW)
os.environ["nnUNet_preprocessed"] = str(NNUNET_PREPROCESSED)
os.environ["nnUNet_results"] = str(NNUNET_RESULTS)

print("RAW:", NNUNET_RAW)
print("PREPROCESSED:", NNUNET_PREPROCESSED)
print("RESULTS:", NNUNET_RESULTS)

RAW: /content/drive/MyDrive/TFM_nnUNet/nnUNet_raw
PREPROCESSED: /content/drive/MyDrive/TFM_nnUNet/nnUNet_preprocessed
RESULTS: /content/drive/MyDrive/TFM_nnUNet/nnUNet_results


In [3]:
folder_3d = (
    NNUNET_PREPROCESSED
    / "Dataset501_ImageCAS"
    / "nnUNetPlans_3d_fullres"
)

images = [
    p for p in folder_3d.glob("*.b2nd")
    if not p.name.endswith("_seg.b2nd")
]

segmentations = list(
    folder_3d.glob("*_seg.b2nd")
)

properties = list(
    folder_3d.glob("*.pkl")
)

print("Imágenes:", len(images))
print("Segmentaciones:", len(segmentations))
print("PKL:", len(properties))

assert len(images) == 200
assert len(segmentations) == 200
assert len(properties) == 200

print("✅ Preprocesado 3D completo.")

Imágenes: 200
Segmentaciones: 200
PKL: 200
✅ Preprocesado 3D completo.


## 1. Operaciones morfológicas diferenciables 3D

Implementación de erosión, dilatación y apertura suaves sobre volúmenes 3D, y del esqueleto diferenciable (soft-skeleton) construido a partir de ellas.


In [4]:
import torch
import torch.nn.functional as F


def soft_erode_3d(x):
    """
    Aproximación diferenciable de una erosión 3D.
    """
    p1 = -F.max_pool3d(
        -x,
        kernel_size=(3, 1, 1),
        stride=1,
        padding=(1, 0, 0)
    )

    p2 = -F.max_pool3d(
        -x,
        kernel_size=(1, 3, 1),
        stride=1,
        padding=(0, 1, 0)
    )

    p3 = -F.max_pool3d(
        -x,
        kernel_size=(1, 1, 3),
        stride=1,
        padding=(0, 0, 1)
    )

    return torch.minimum(
        torch.minimum(p1, p2),
        p3
    )


def soft_dilate_3d(x):
    """
    Aproximación diferenciable de una dilatación 3D.
    """
    return F.max_pool3d(
        x,
        kernel_size=3,
        stride=1,
        padding=1
    )


def soft_open_3d(x):
    """
    Apertura morfológica:
    erosión + dilatación.
    """
    return soft_dilate_3d(
        soft_erode_3d(x)
    )


def soft_skeletonize_3d(x, iterations=10):
    """
    Aproximación diferenciable del esqueleto 3D.
    """

    opened = soft_open_3d(x)

    skeleton = F.relu(
        x - opened
    )

    for _ in range(iterations):

        x = soft_erode_3d(x)

        opened = soft_open_3d(x)

        delta = F.relu(
            x - opened
        )

        skeleton = (
            skeleton
            + F.relu(
                delta - skeleton * delta
            )
        )

    return skeleton

## 2. SoftClDiceLoss

Pérdida basada en la precisión y la sensibilidad topológicas entre los esqueletos suaves de la predicción y del ground truth.


In [5]:
import torch.nn as nn


class SoftClDiceLoss(nn.Module):

    def __init__(
        self,
        iterations=10,
        smooth=1e-5
    ):
        super().__init__()

        self.iterations = iterations
        self.smooth = smooth


    def forward(
        self,
        prediction,
        target
    ):

        # Cálculo en FP32 para mayor estabilidad
        prediction = prediction.float()
        target = target.float()

        # Esqueleto de la predicción
        skel_pred = soft_skeletonize_3d(
            prediction,
            iterations=self.iterations
        )

        # Esqueleto del ground truth
        skel_true = soft_skeletonize_3d(
            target,
            iterations=self.iterations
        )

        # Precisión topológica
        tprec = (
            (skel_pred * target).sum()
            + self.smooth
        ) / (
            skel_pred.sum()
            + self.smooth
        )

        # Sensibilidad topológica
        tsens = (
            (skel_true * prediction).sum()
            + self.smooth
        ) / (
            skel_true.sum()
            + self.smooth
        )

        # clDice
        cldice = (
            2.0 * tprec * tsens
        ) / (
            tprec
            + tsens
            + self.smooth
        )

        # Se minimiza la pérdida (1 - clDice)
        return 1.0 - cldice

## 3. Combinación Dice + CE + soft-clDice

La pérdida total suma la pérdida regional original de nnU-Net (Dice + CE) y el término soft-clDice ponderado por λ.


In [6]:
class DiceCEPlusSoftClDice(nn.Module):

    def __init__(
        self,
        regional_loss,
        lambda_cldice=0.2,
        skeleton_iterations=10
    ):
        super().__init__()

        self.regional_loss = regional_loss
        self.lambda_cldice = lambda_cldice

        self.soft_cldice = SoftClDiceLoss(
            iterations=skeleton_iterations,
            smooth=1e-5
        )


    def forward(
        self,
        logits,
        target
    ):

        # 1. Pérdida regional original de nnU-Net (Dice + Cross-Entropy)
        loss_regional = self.regional_loss(
            logits,
            target
        )

        # 2. Conversión de logits a probabilidades
        probabilities = torch.softmax(
            logits.float(),
            dim=1
        )

        # ImageCAS: canal 0 = fondo, canal 1 = coronarias
        foreground_probability = (
            probabilities[:, 1:2]
        )

        # 3. Adaptación de la dimensión del target
        if target.ndim == logits.ndim - 1:
            target = target.unsqueeze(1)

        # 4. Ground truth binario
        foreground_target = (
            target == 1
        ).float()

        # 5. Pérdida topológica
        topology_loss = self.soft_cldice(
            foreground_probability,
            foreground_target
        )

        # 6. Pérdida total
        total_loss = (
            loss_regional
            +
            self.lambda_cldice
            * topology_loss
        )

        return total_loss

## 4. Clase base del trainer `nnUNetTrainerSoftClDice100epochs`

Definición del trainer, con el peso del término topológico (`lambda_cldice`) y el número de iteraciones de skeletonization como atributos.


In [7]:
import numpy as np
import torch

from nnunetv2.training.loss.compound_losses import (
    DC_and_CE_loss
)

from nnunetv2.training.loss.deep_supervision import (
    DeepSupervisionWrapper
)

from nnunetv2.training.loss.dice import (
    MemoryEfficientSoftDiceLoss
)

from nnunetv2.training.nnUNetTrainer.variants.training_length.nnUNetTrainer_Xepochs import (
    nnUNetTrainer_100epochs
)


class nnUNetTrainerSoftClDice100epochs(
    nnUNetTrainer_100epochs
):

    def __init__(
        self,
        plans,
        configuration,
        fold,
        dataset_json,
        device=torch.device("cuda")
    ):

        super().__init__(
            plans,
            configuration,
            fold,
            dataset_json,
            device
        )

        # Peso del término topológico
        self.lambda_cldice = 0.2

        # Número de iteraciones de skeletonization
        self.skeleton_iterations = 10

## 5. Sobrescritura de `_build_loss()`

Se reemplaza la construcción de la pérdida del trainer estándar por la combinación Dice + CE + soft-clDice, manteniendo el esquema de deep supervision de nnU-Net.


In [8]:
def custom_build_loss(self):

    # 1. Pérdida regional original de nnU-Net

    regional_loss = DC_and_CE_loss(

        soft_dice_kwargs={
            "batch_dice":
                self.configuration_manager.batch_dice,

            "smooth": 1e-5,

            "do_bg": False,

            "ddp": self.is_ddp
        },

        ce_kwargs={},

        weight_ce=1,

        weight_dice=1,

        ignore_label=
            self.label_manager.ignore_label,

        dice_class=
            MemoryEfficientSoftDiceLoss
    )


    # 2. Término soft-clDice

    loss = DiceCEPlusSoftClDice(

        regional_loss=
            regional_loss,

        lambda_cldice=
            self.lambda_cldice,

        skeleton_iterations=
            self.skeleton_iterations
    )


    # 3. Deep supervision de nnU-Net

    if self.enable_deep_supervision:

        deep_supervision_scales = (
            self._get_deep_supervision_scales()
        )

        weights = np.array([
            1 / (2 ** i)
            for i in range(
                len(deep_supervision_scales)
            )
        ])

        # La salida de menor resolución no contribuye a la pérdida final
        weights[-1] = 0

        weights = (
            weights
            / weights.sum()
        )

        loss = DeepSupervisionWrapper(
            loss,
            weights
        )


    return loss


# Sustitución del método de la clase
nnUNetTrainerSoftClDice100epochs._build_loss = (
    custom_build_loss
)

## 6. Test de gradientes

Antes de guardar el trainer y lanzar las 100 épocas, se comprueba con tensores sintéticos que soft-clDice admite retropropagación.


In [9]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

# Predicción sintética
pred_logits = torch.randn(
    1,
    1,
    24,
    32,
    32,
    device=device,
    requires_grad=True
)

pred_probability = torch.sigmoid(
    pred_logits
)

# Ground truth sintético
target_test = torch.zeros_like(
    pred_probability
)

# Estructura tubular simulada
target_test[
    :,
    :,
    4:20,
    15:17,
    15:17
] = 1


# Cálculo de soft-clDice
test_loss = SoftClDiceLoss(
    iterations=5
)(
    pred_probability,
    target_test
)

print(
    "soft-clDice loss:",
    test_loss.item()
)


# Retropropagación
test_loss.backward()


print(
    "Gradiente calculado:",
    pred_logits.grad is not None
)

print(
    "Gradientes finitos:",
    torch.isfinite(
        pred_logits.grad
    ).all().item()
)

soft-clDice loss: 0.9951531291007996
Gradiente calculado: True
Gradientes finitos: True


In [13]:
from pathlib import Path
import os

BASE = Path("/content/drive/MyDrive/TFM_nnUNet")

CUSTOM_TRAINERS = BASE / "custom_trainers"

CUSTOM_TRAINERS.mkdir(
    parents=True,
    exist_ok=True
)

print("Carpeta creada:")
print(CUSTOM_TRAINERS)

Carpeta creada:
/content/drive/MyDrive/TFM_nnUNet/custom_trainers


In [14]:
trainer_code = r'''
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from nnunetv2.training.loss.compound_losses import DC_and_CE_loss
from nnunetv2.training.loss.deep_supervision import DeepSupervisionWrapper
from nnunetv2.training.loss.dice import MemoryEfficientSoftDiceLoss

from nnunetv2.training.nnUNetTrainer.variants.training_length.nnUNetTrainer_Xepochs import (
    nnUNetTrainer_100epochs
)


def soft_erode_3d(x):

    p1 = -F.max_pool3d(
        -x,
        kernel_size=(3, 1, 1),
        stride=1,
        padding=(1, 0, 0)
    )

    p2 = -F.max_pool3d(
        -x,
        kernel_size=(1, 3, 1),
        stride=1,
        padding=(0, 1, 0)
    )

    p3 = -F.max_pool3d(
        -x,
        kernel_size=(1, 1, 3),
        stride=1,
        padding=(0, 0, 1)
    )

    return torch.minimum(
        torch.minimum(p1, p2),
        p3
    )


def soft_dilate_3d(x):

    return F.max_pool3d(
        x,
        kernel_size=3,
        stride=1,
        padding=1
    )


def soft_open_3d(x):

    return soft_dilate_3d(
        soft_erode_3d(x)
    )


def soft_skeletonize_3d(x, iterations=10):

    opened = soft_open_3d(x)

    skeleton = F.relu(
        x - opened
    )

    for _ in range(iterations):

        x = soft_erode_3d(x)

        opened = soft_open_3d(x)

        delta = F.relu(
            x - opened
        )

        skeleton = (
            skeleton
            + F.relu(
                delta - skeleton * delta
            )
        )

    return skeleton


class SoftClDiceLoss(nn.Module):

    def __init__(
        self,
        iterations=10,
        smooth=1e-5
    ):
        super().__init__()

        self.iterations = iterations
        self.smooth = smooth


    def forward(
        self,
        prediction,
        target
    ):

        prediction = prediction.float()
        target = target.float()

        skel_pred = soft_skeletonize_3d(
            prediction,
            iterations=self.iterations
        )

        skel_true = soft_skeletonize_3d(
            target,
            iterations=self.iterations
        )

        tprec = (
            (skel_pred * target).sum()
            + self.smooth
        ) / (
            skel_pred.sum()
            + self.smooth
        )

        tsens = (
            (skel_true * prediction).sum()
            + self.smooth
        ) / (
            skel_true.sum()
            + self.smooth
        )

        cldice = (
            2.0 * tprec * tsens
        ) / (
            tprec
            + tsens
            + self.smooth
        )

        return 1.0 - cldice


class DiceCEPlusSoftClDice(nn.Module):

    def __init__(
        self,
        regional_loss,
        lambda_cldice=0.2,
        skeleton_iterations=10
    ):
        super().__init__()

        self.regional_loss = regional_loss
        self.lambda_cldice = lambda_cldice

        self.soft_cldice = SoftClDiceLoss(
            iterations=skeleton_iterations,
            smooth=1e-5
        )


    def forward(
        self,
        logits,
        target
    ):

        loss_regional = self.regional_loss(
            logits,
            target
        )

        probabilities = torch.softmax(
            logits.float(),
            dim=1
        )

        foreground_probability = (
            probabilities[:, 1:2]
        )

        if target.ndim == logits.ndim - 1:
            target = target.unsqueeze(1)

        foreground_target = (
            target == 1
        ).float()

        topology_loss = self.soft_cldice(
            foreground_probability,
            foreground_target
        )

        return (
            loss_regional
            +
            self.lambda_cldice
            * topology_loss
        )


class nnUNetTrainerSoftClDice100epochs(
    nnUNetTrainer_100epochs
):

    def __init__(
        self,
        plans,
        configuration,
        fold,
        dataset_json,
        device=torch.device("cuda")
    ):

        super().__init__(
            plans,
            configuration,
            fold,
            dataset_json,
            device
        )

        self.lambda_cldice = 0.2
        self.skeleton_iterations = 10


    def _build_loss(self):

        regional_loss = DC_and_CE_loss(

            soft_dice_kwargs={
                "batch_dice":
                    self.configuration_manager.batch_dice,

                "smooth": 1e-5,

                "do_bg": False,

                "ddp": self.is_ddp
            },

            ce_kwargs={},

            weight_ce=1,

            weight_dice=1,

            ignore_label=
                self.label_manager.ignore_label,

            dice_class=
                MemoryEfficientSoftDiceLoss
        )


        loss = DiceCEPlusSoftClDice(

            regional_loss=
                regional_loss,

            lambda_cldice=
                self.lambda_cldice,

            skeleton_iterations=
                self.skeleton_iterations
        )


        if self.enable_deep_supervision:

            deep_supervision_scales = (
                self._get_deep_supervision_scales()
            )

            weights = np.array([
                1 / (2 ** i)
                for i in range(
                    len(deep_supervision_scales)
                )
            ])

            weights[-1] = 0

            weights = (
                weights
                / weights.sum()
            )

            loss = DeepSupervisionWrapper(
                loss,
                weights
            )


        return loss
'''

trainer_file = (
    CUSTOM_TRAINERS
    / "nnUNetTrainerSoftClDice100epochs.py"
)

trainer_file.write_text(trainer_code)

print("Guardado en:")
print(trainer_file)

Guardado en:
/content/drive/MyDrive/TFM_nnUNet/custom_trainers/nnUNetTrainerSoftClDice100epochs.py


In [15]:
os.environ["nnUNet_extTrainer"] = str(CUSTOM_TRAINERS)

print(os.environ["nnUNet_extTrainer"])

/content/drive/MyDrive/TFM_nnUNet/custom_trainers


In [16]:
!echo $nnUNet_extTrainer

/content/drive/MyDrive/TFM_nnUNet/custom_trainers


## 7. Registro del trainer

El comando `nnUNetv2_train` no localiza las clases definidas únicamente en memoria. Por ello, el código del trainer se escribe en un archivo `.py` dentro de una carpeta propia en Drive, que se registra mediante la variable de entorno `nnUNet_extTrainer`.


In [10]:
from pathlib import Path
import os

CUSTOM_TRAINERS = (
    Path("/content/drive/MyDrive/TFM_nnUNet")
    / "custom_trainers"
)

CUSTOM_TRAINERS.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Carpeta trainers:",
    CUSTOM_TRAINERS
)

Carpeta trainers: /content/drive/MyDrive/TFM_nnUNet/custom_trainers


## 8. Entrenamiento soft-clDice

Entrenamiento con el trainer registrado en la sección anterior, mediante el comando:


In [17]:
!nnUNetv2_train \
501 \
3d_fullres \
0 \
-tr nnUNetTrainerSoftClDice100epochs \
--npz


############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Trainer 'nnUNetTrainerSoftClDice100epochs' not found in nnunetv2.training.nnUNetTrainer.
Searching in external trainer paths from environment variable 'nnUNet_extTrainer'...
Searching in: /content/drive/MyDrive/TFM_nnUNet/custom_trainers
Searching for class nnUNetTrainerSoftClDice100epochs in folder /content/drive/MyDrive/TFM_nnUNet/custom_trainers with current module None
  Inspecting module: nnUNetTrainerSoftClDice100epochs
Found class nnUNetTrainerSoftClDice100epochs in nnUNetTrainerSoftClDice100epochs
Using trainer 'nnUNetTrainerSoftClDice100epochs' from: /content/drive/MyDrive/TFM_nnUNet/custom_trainers
Using device: cuda:0

#######################################################################
Please cit